# ML Project Template

End-to-end workflow:  
**EDA → Preprocessing → Feature Selection → Model Building → Evaluation → Tuning → Pipeline**

Use this notebook for any classification or regression project.  
Replace example code with your own data and logic.

## 1. Problem & Data Overview

**Why this step:**
- Define the task: classification, regression, or clustering?
- Understand what we’re predicting and how we’ll measure success.
- Check data quality: missing values, outliers, wrong types.

In [ ]:
# Load data
import pandas as pd
import numpy as np

# 🚨 Replace with your own dataset path (e.g., "my_data.csv")
df = pd.read_csv("your_dataset.csv")

# Basic inspection
print(df.head())
print(df.info())
print(df.describe())

# Check missing values
print(df.isnull().sum())

# Define your target column name here
target = "your_target_column"
X = df.drop(columns=[target])
y = df[target]

## 2. Exploratory Data Analysis (EDA)

**Why EDA:**
- Understand distributions, relationships, and biases.
- Detect outliers, skewed features, and categorical patterns.
- Inform preprocessing and feature engineering choices.

In [ ]:
# You can add your EDA code here
import matplotlib.pyplot as plt
import seaborn as sns

# Example: histogram of a numeric feature
# sns.histplot(df["age"], kde=True)
# plt.title("Age distribution")
# plt.show()

# Example: countplot of a categorical feature
# sns.countplot(data=df, x="gender")
# plt.title("Gender distribution")
# plt.show()

# Example: correlation heatmap (numeric only)
# num_df = df.select_dtypes(include=[np.number])
# corr = num_df.corr()
# sns.heatmap(corr, annot=False, cmap="coolwarm")
# plt.title("Feature correlation")
# plt.show()

# Example: target vs feature
# sns.boxplot(data=df, x="gender", y="age", hue="churn")
# plt.show()

## 3. Data Preprocessing

**Why preprocessing:**
- Models need clean, numeric, consistent data.
- Missing values break many algorithms.
- Scaled features help gradient-based and distance-based models.
- Categorical variables must be encoded.

**Steps:**
1. Handle missing values.
2. Remove duplicates / obvious noise.
3. Encode categorical variables.
4. Handle outliers (optional).
5. Scale features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Split before preprocessing (to avoid leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Identify types
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Impute missing values
imputer = SimpleImputer(strategy="mean")  # or "median", "most_frequent"

# Encode categorical
ohe = OneHotEncoder(handle_unknown="ignore")

# Scale numeric
scaler = StandardScaler()

# Combine with ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num_impute", imputer, numeric_cols),
        ("num_scale", scaler, numeric_cols),
        ("cat_encode", ohe, cat_cols),
    ],
    remainder="drop"
)

# Fit on train, transform train & test
X_train_proc = preprocess.fit_transform(X_train)
X_test_proc = preprocess.transform(X_test)

## 4. Feature Selection

**Why feature selection:**
- Reduce noise and overfitting.
- Improve model speed and interpretability.
- Remove redundant or irrelevant features.

**Common methods:**
- Filter methods: correlation, variance threshold, statistical tests.
- Model-based: feature importance from trees.
- Wrapper methods: RFE, SelectFromModel.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import VarianceThreshold

# A. Variance threshold (remove near-constant features)
vt = VarianceThreshold(threshold=1e-4)
X_train_fs = vt.fit_transform(X_train_proc)
X_test_fs = vt.transform(X_test_proc)

# B. SelectKBest (statistical test)
selector_kbest = SelectKBest(score_func=f_classif, k=10)
X_train_fs = selector_kbest.fit_transform(X_train_fs, y_train)
X_test_fs = selector_kbest.transform(X_test_fs)

# C. Model-based selection (tree importance)
tree_base = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_base.fit(X_train_fs, y_train)

model_sel = SelectFromModel(tree_base, threshold="mean")
X_train_sel = model_sel.fit_transform(X_train_fs, y_train)
X_test_sel = model_sel.transform(X_test_fs)

# Inspect selected features correctly
selected_mask = model_sel.get_support()
all_features = preprocess.get_feature_names_out()
selected_features = all_features[selected_mask]

print("Selected Features:", selected_features)

## 5. Model Building

**Why model building:**
- Train models to learn patterns from features.
- Use multiple models and compare performance.
- Avoid overfitting via cross-validation and proper splits.

**Common models:**
- Classification: Logistic Regression, Random Forest, SVM, Gradient Boosting.
- Regression: Linear Regression, Ridge, Lasso, Random Forest Regressor.

In [ ]:
# You can add your model building code here
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Example: classification
models = {
    "log_reg": LogisticRegression(max_iter=1000),
    "rf": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    "svc": SVC(),
    "dt": DecisionTreeClassifier(max_depth=4, random_state=42),
}

for name, model in models.items():
    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    print(name, "accuracy:", model.score(X_test_sel, y_test))

## 6. Model Evaluation

**Why evaluation:**
- Measure how well the model generalizes.
- Compare models objectively.
- Detect overfitting / underfitting.

**Metrics:**
- Classification: accuracy, precision, recall, F1, ROC-AUC, confusion matrix.
- Regression: MSE, RMSE, MAE, R2.

In [ ]:
# You can add your evaluation code here
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    classification_report
)

# Best model (example: Random Forest)
best_model = models["rf"]
y_pred = best_model.predict(X_test_sel)
y_proba = best_model.predict_proba(X_test_sel)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred))

## 7. Cross-Validation & Hyperparameter Tuning

**Why CV and tuning:**
- Cross-validation: better estimate of model performance.
- Tuning: find optimal hyperparameters to improve accuracy and reduce overfitting.

In [ ]:
# You can add your CV and tuning code here
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold

# Cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_sel, y_train, cv=kf)
print("CV scores:", cv_scores)
print("Mean CV accuracy:", cv_scores.mean())
print("Std CV accuracy:", cv_scores.std())

# Hyperparameter tuning (example: Random Forest)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6, 8],
    "min_samples_split": [2, 5],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_sel, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

tuned_model = grid.best_estimator_
y_pred_tuned = tuned_model.predict(X_test_sel)
print("Tuned accuracy:", accuracy_score(y_test, y_pred_tuned))

## 8. Full Pipeline (Preprocessing + Selection + Model)

**Why pipelines:**
- Avoid code duplication.
- Ensure train/test processing is consistent.
- Make deployment and retraining easier.

In [ ]:
# You can add your pipeline code here
from sklearn.pipeline import Pipeline

full_pipeline = Pipeline([
    ("preprocess", preprocess),
    # If you want to include feature selection inside pipeline,
    # you can add a Selector step here.
    ("model", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
])

# On raw X (not preprocessed)
full_pipeline.fit(X_train, y_train)
y_pred_pipe = full_pipeline.predict(X_test)

print("Pipeline accuracy:", accuracy_score(y_test, y_pred_pipe))

## 9. Saving & Loading Models

**Why save/load:**
- Reuse models without retraining.
- Deploy models in production.

In [ ]:
# You can add your save/load code here
import joblib

# Save
# joblib.dump(tuned_model, "model_random_forest.joblib")
# joblib.dump(preprocess, "preprocess.joblib")

# Load
# loaded_model = joblib.load("model_random_forest.joblib")
# loaded_preprocess = joblib.load("preprocess.joblib")

# Use
# y_new = loaded_model.predict(loaded_preprocess.transform(new_X))

## 10. Quick Summary of Steps & Why

1. **Problem definition** – Clarify task and success metrics.
2. **Data loading & inspection** – Understand structure, types, missing values.
3. **EDA** – Discover patterns, outliers, and relationships.
4. **Preprocessing** – Clean, encode, and scale data for models.
5. **Feature selection** – Reduce noise, improve speed, and reduce overfitting.
6. **Model building** – Learn patterns from features.
7. **Evaluation** – Measure generalization and compare models.
8. **Cross-validation & tuning** – Robust performance estimate and better hyperparameters.
9. **Pipeline & saving** – Consistency, reusability, and deployment.

**One-line memory hook:**  
ML workflow = problem → data → EDA → preprocess → feature selection → model → evaluate → tune → deploy.